In [ ]:
import os
import sys
import glob

# --- 1. AUTO-DETECT JAVA IN CONDA ENV ---
# Get the path to your current 'big_data' environment
current_env = sys.prefix 

# Look for java.exe in the standard Conda locations
possible_java_homes = [
    os.path.join(current_env, "Library", "lib", "jvm"),  # Standard Conda location
    os.path.join(current_env, "Library"),                # Alternative
    os.path.join(current_env)                            # Fallback
]

java_home = None
for path in possible_java_homes:
    # We are looking for a folder that contains "bin/java.exe"
    # Sometimes it's deep inside, so we look for subfolders starting with "openjdk" or "jdk"
    search_pattern = os.path.join(path, "**", "bin", "java.exe")
    found = glob.glob(search_pattern, recursive=True)
    
    if found:
        # The JAVA_HOME is the folder containing the 'bin' folder
        java_bin = found[0]
        java_home = os.path.dirname(os.path.dirname(java_bin))
        print(f"✅ Found Java at: {java_home}")
        break

if not java_home:
    raise Exception("❌ Could not find Java in your Conda environment. Did you run 'conda install openjdk=11'?")

# --- 2. CONFIGURE ENVIRONMENT ---
os.environ['JAVA_HOME'] = java_home
os.environ['HADOOP_HOME'] = r"C:\hadoop"
sys.path.append(r"C:\hadoop\bin")

# Add paths to System PATH variable
os.environ['PATH'] = os.path.join(java_home, "bin") + ";" + r"C:\hadoop\bin" + ";" + os.environ['PATH']

# --- 3. START SPARK ---
from pyspark.sql import SparkSession

print("⏳ Starting Spark Session with Auto-Detected Java...")

spark = SparkSession.builder \
    .appName("IngestFlightData") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print("🚀 Spark initialized successfully!")

In [ ]:
from pyspark.sql import SparkSession

# 1. Initialize Spark
spark = SparkSession.builder.appName("IngestFlightData").getOrCreate()

# 2. Define the paths using wildcards (*)
# Assuming the CSV files are in the same folder as your notebook. 
# If they are in a subfolder called 'data', add that (e.g., "data/report_2017_*.csv")
files_to_load = [
    r"C:\Users\nhatp\OneDrive - NOVAIMS\Desktop\this semester\projects\big_data_analytic_project\csv_flight\report_2017_*.csv",  # Loads all 2017 months
    r"C:\Users\nhatp\OneDrive - NOVAIMS\Desktop\this semester\projects\big_data_analytic_project\csv_flight\report_2018_*.csv"   # Loads all 2018 months
]

flight_schema = StructType([
    StructField("FL_DATE", DateType(), True),
    StructField("OP_CARRIER", StringType(), True),
    StructField("OP_CARRIER_FL_NUM", IntegerType(), True),
    StructField("ORIGIN", StringType(), True),
    StructField("DEST", StringType(), True),
    StructField("CRS_DEP_TIME", IntegerType(), True), # Scheduled Dep Time
    StructField("DEP_TIME", FloatType(), True),       # Actual Dep Time
    StructField("DEP_DELAY", FloatType(), True),
    StructField("TAXI_OUT", FloatType(), True),
    StructField("WHEELS_OFF", FloatType(), True),
    StructField("WHEELS_ON", FloatType(), True),
    StructField("TAXI_IN", FloatType(), True),
    StructField("CRS_ARR_TIME", IntegerType(), True),
    StructField("ARR_TIME", FloatType(), True),
    StructField("ARR_DELAY", FloatType(), True),
    StructField("CANCELLED", FloatType(), True),
    StructField("CANCELLATION_CODE", StringType(), True),
    StructField("DIVERTED", FloatType(), True),
    StructField("CRS_ELAPSED_TIME", FloatType(), True),
    StructField("ACTUAL_ELAPSED_TIME", FloatType(), True),
    StructField("AIR_TIME", FloatType(), True),
    StructField("DISTANCE", FloatType(), True),
    StructField("CARRIER_DELAY", FloatType(), True),
    StructField("WEATHER_DELAY", FloatType(), True),
    StructField("NAS_DELAY", FloatType(), True),
    StructField("SECURITY_DELAY", FloatType(), True),
    StructField("LATE_AIRCRAFT_DELAY", FloatType(), True)
])

# 3. Read the data with your Explicit Schema 

df_raw = spark.read.csv(files_to_load, header=True, schema=flight_schema)

# 4. Verify you only have 2017 and 2018
from pyspark.sql.functions import year
df_raw.select(year("FL_DATE").alias("Year")).distinct().show()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, to_timestamp, year, month, when

# Initialize Spark Session (Big Data Safe: standard config)
spark = SparkSession.builder \
    .appName("FlightDataIngestion") \
    .getOrCreate()